***
# Mapping Wildfires
#### by Paul Ghisletti, as part of the SDS210 course (spring semester '26)

***

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path
import os
import re
import sys
import unicodedata
import warnings
import webbrowser

from dotenv import load_dotenv
import branca.colormap as cm
import folium as fm
from folium import plugins
import geopandas as gpd
import numpy as np
import pandas as pd
import requests
from sklearn.cluster import DBSCAN

In [ ]:
%load_ext autoreload
%autoreload 2

In order to import packages from the `src/` folder, we need to set the root of this notebook to the repository's root folder:

In [ ]:
sys.path.append(str(Path().resolve().parent))

***
## API-Setup
Make sure you followed the steps 2.1. to 2.3. in the `README.md` file on how to set your individual API-key: Check, whether the following output matches your individual API-key from FIRMS. Here, you can also see how many free transactions you have left with your api-key

In [ ]:
# load the content of the `.env` file
load_dotenv()

# assigns the api-keay which is specified in the `.env` file to variable
api_key = os.getenv("MY_API_KEY")
print(f"API-Key: {api_key}")

# uses FIRMS api to get information on transactions
url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + api_key
try:
  df = pd.Series(requests.get(url).json()) # gets a table of information about your api key usage
  display(df)
except:
  print ("There is an issue with your API key. Please check the value of MY_API_KEY in your .env file and your internet connection and try again.")

We can also define a function which tells us how many transactions we have used so far. This can be used to check, how many transactions one API query costs (code from the [FIRMS notebook on *API use*](https://firms.modaps.eosdis.nasa.gov/academy/data_api/)):

In [ ]:
def get_transaction_count(api_key = api_key) :
  count = 0
  url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + api_key
  try:
    response = requests.get(url)
    data = response.json()
    df = pd.Series(data)
    count = df['current_transactions']
  except:
    print ("Error in our call.")
  return count

tcount = get_transaction_count()
print (f'Our current transaction count is {tcount}')

***
## 1. Loading a dataset through the API
### 1.1. Preparations
#### 1.1.1. Loading country and continent outlines
Because we will later on enable the user to simply type the continent or country for the desired extent, we need a dataset that stores the bounding box coordinates which we can reference the user input against. This is done by importing a `.gpkg` file with all 198 countries of the world (193 UN members, 2 UN observer states, 3 disputed states) and one for the 7 continents. 
First, we import the `.gpkg` files as `GeoDataFrame`s, which we created from `ne_10m_admin_0_countries.shp` and `continent_boundaries_7.gpkg`respectively in the dedicated Jupyter Notebook (`/notebooks/countries_normalisation.ipynb`).

In [ ]:
countries_gdf = gpd.read_file("../data/raw/198_countries.gpkg").to_crs(epsg=4326)
continents_gdf = gpd.read_file("../data/raw/7_continents.gpkg").to_crs(epsg=4326)

#### 1.1.2. DataClass for API output
By handling the API query input and output as a Class object, we can later easily access metadata or even define methods (e.g. .get_coordinates()). For this to work, we need to set up a DataClass:

In [ ]:
from dataclasses import dataclass

@dataclass
class WildFireQuery:
    data: gpd.GeoDataFrame
    sensor: str
    area: str
    extent: str
    date: str
    n_days: int
    sampled: bool = False
    cleaned: gpd.GeoDataFrame = None
    clustered: gpd.GeoDataFrame = None
    geometry: gpd.GeoDataFrame = None

    def __post_init__(self):
        # deduces instrument from sensor
        sensor_list = self.sensor.split("_")
        self.instrument = sensor_list[0]
        
        # calculates pixel area$
        if self.instrument in ["VIIRS", "MODIS"]:
            self.pixel_area = self.data["track"].mean() * self.data["scan"].mean()
        if self.instrument == 'LANDSAT':
            self.pixel_area = 0.03 ** 2

        # handles the special case of 'world'
        if self.area == "world":
            self.extent = "-180,-90,180,90"
        self.bbox = [float(n) for n in self.extent.split(",")]

        # display name, because it is lost during the api query
        if self.area == "world":
            self.area_display = "World"

        # we can use the gdf for continents and countries to look up the names again
        elif self.area in continents_gdf["CONTINENT_NORM"].values:
            match = continents_gdf[
                continents_gdf["CONTINENT_NORM"] == self.area
            ]

            self.area_display = (
                match["CONTINENT"].values[0]
                if not match.empty
                else self.area
            )

        else:
            match = countries_gdf[
                countries_gdf["name"] == self.area
            ]

            self.area_display = (
                match["ADMIN"].values[0]
                if not match.empty
                else self.area
            )
        # epsilon for dbscan depending on instrument
        if self.instrument == "MODIS":
            self.dbscan_epsilon = 2
        elif self.instrument == "VIIRS":
            self.dbscan_epsilon = 1.3
        elif self.instrument == "LANDSAT":
            self.dbscan_epsilon = 0.1

    # calculate coordinates of polygon center for centering the final folium map
    def get_center(self) -> tuple:
        if self.area == "world":
            return (20, 0)
        bbox = self.bbox
        return ((bbox[1] + bbox[3]) / 2,
                (bbox[0] + bbox[2]) / 2)
    
    # calculate fitting start zoom based on area extent for final folium map
    def get_zoom_level(self) -> int:
        bbox = self.bbox
        max_extent = max(bbox[2] - bbox[0],
                         bbox[3] - bbox[1])

        # thresholds were picked manually 
        if max_extent >= 180: return 2
        elif max_extent > 60: return 3
        elif max_extent > 30: return 4
        elif max_extent > 18: return 5
        elif max_extent > 10: return 6
        elif max_extent > 5:  return 7
        else:                 return 8

#### 1.1.3. Check availability
Because the latency differs between the sensors (actually, it mostly changes between different products of the same sensor provided by FIRMS. The -SP products have been processed by FIRMS extensively and contain the extra variabla `type`. This takes some time, which is why the latency for those outputs is usually a couple of months, whereas the latency for -NRT products is almost instantaneous), we first need to check the date availability. We can later use this information when automating the download.  

This function is important for later use, where we want to automate the process of getting data from the API. Since the user usually does not know the date of the latest available data, we need to look this up automatically, which is where this function comes into play.

In [ ]:
def get_availability_all(api_key):
    url = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + api_key + '/ALL'
    return pd.read_csv(url) 

availability_all_df = get_availability_all(api_key)
display(availability_all_df.head(10))

#### 1.1.4. Overview Function
If we want to quickly look at what our gdf contains, how many unique values each column has and so forth, we can use the funcrion defined in the following chunk of code:

In [ ]:
def overview(df, name="dataset"):
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_unique": df.nunique(dropna=True)
    }).sort_values(["dtype", "n_unique"], ascending=[True, False])

    print(f"\n=== {name} ===")
    print("Shape:", df.shape)
    display(summary)

### 1.2. API Query
In this chapter, we will define a function which will automatically get the API data. All the user has to do is provide the following parameters:
- The desired sensor
- The area which the data should cover
- The date of data acquisition
- Time frame of the data  

Since this is a function where we need to check numerous parameter inputs, it is helpful to define helper functions to make the final function more readable. These helpers mainly check for correct input format and type.  
Since there is a lot going on, I will describe the steps performed by the function in words below:
- It checks for each input parameter, whether it is a valid input by the user, raising a `ValueError` otherwise.
    - Sensor: Must be one of the available sensors from the FIRMS API.
    - Area: explained below.
    - Date: valid string format: YYYY-MM-DD and it is within the provided range of dates by the FIRMS API.
    - N_days: at least 1 but only up to 5 days in one query.
- It uses the `ALIASES` dictionary from `/data/raw` (which was defined by generative AI) in a normalisation function to map user input to a valid country name. This accepts a wide range of alternative spellings and handles special characters, empty spaces and punctuation. It is worth noting that this also works with continents and "world".
- Using the `countries_gdf` or `continents_gdf` from above, which contain the normalised country and continent names as well as their geometry, it gets the bounding coordinates of the query extent. For continents it uses a dictionary where the 7 coordinate lists are predefined.
- Handling the date is also non-trivial, since the user can leave this parameter empty if they wish to get the latest data. Therefore, we need to consult the data frame which we fetched from the FIRMS API a couple steps above. This data frame contains the latest available date for each of the sensors. My function then either uses this date or the date provided by the user, given it is older than the latest possible date, raising a `ValueError` otherwise.
- It then constructs the API-URL which is how we pass all the input information to the FIRMS API.
- With the URL we can get the data which is directly converted from a `pd.DataFrame` to a `gpd.GeoDataFrame` with the `lat` and `lon` columns.

- Before finalising the output, we clip the data points to the geometry of the data extent defined by the user (e.g. a country or an entire continent). This prevents confusing outputs when looking for wildfires data in France, since the bounding box of the France geometry basically covers the entire globe with all its overseas territories.
- Both the newly calculated output and the user input is then passed into a new `WildFireQuery` class object.

In [ ]:
# -----------------------------------------------
# Helper Functions for checking parameter inputs:
# -----------------------------------------------
def check_sensor(sensor):
    valid_sensors = set(availability_all_df["data_id"]) 
    if sensor not in valid_sensors:
        raise ValueError("Sensor: Invalid sensor name.")

def check_area_list(area):
    west, south, east, north = area
    # check invalid longitude values
    if west < -180 or west > 180: 
        raise ValueError("Area: Invalid coordinate: west")
    if east < -180 or east > 180: 
        raise ValueError("Area: Invalid coordinate: east")
    # check invalid latitude values
    if south < -90 or south > 90:
        raise ValueError("Area: Invalid coordinate: south")
    if north < -90 or north > 90:
        raise ValueError("Area: Invalid coordinate: north")
    # check order of list
    if west > east:
        raise ValueError("Area: Invalid coordinate: west is larger than east")
    if south > north:
        raise ValueError("Area: Invalid coordinate: south is larger than north")
    
def check_date_str(date, sensor, max_date_time, min_date_time):
    is_none = date is None
    is_str = type(date) == str

    # get 
    if not is_none and not is_str:
        raise ValueError("Date: Invalid input type. Either str or None")
    
    if is_str:
        try:
            date_time = datetime.strptime(date, "%Y-%m-%d")
        except ValueError:
            raise ValueError("Date: Must be in YYYY-MM-DD format and be a valid calendar date")
        if date_time > max_date_time:
            raise ValueError(f"Date: Desired date not available for {sensor}. Latest available data from {max_date_time}")
        if date_time < min_date_time:
            raise ValueError(f"Date: Desired date not available for {sensor}. Earliest available data from {min_date_time}")
        
def check_n_days(n_days):
    if not type(n_days) == int:
        raise ValueError("N_days: Type must be int.")
    if n_days < 1 or n_days > 5:
        raise ValueError("N_days: Minimum 1 day, maximum 5 days.")

# --------------------------------------------------------------------
# Helper Functions for calculating area based on country or continent:
# --------------------------------------------------------------------

# import aliases dictionary from src folder
from src.country_continent_aliases import ALIASES

# function for normalising a country string
def normalise(country: str) -> str:
    country = unicodedata.normalize("NFD", country) # decomposes special characters: ô → o + ^
    country = "".join(c for c in country if unicodedata.category(c) != "Mn") # drop all accents (diacritic characters)
    country = country.lower() # lowercase everything
    country = re.sub(r"[^a-z0-9\s]", "", country) # removes forbidden characters
    country = re.sub(r"\s+", "", country) # removes multi-spaces in middle and all spaces at beginning and end
    country = ALIASES.get(country, country)
    return country

def is_country(area: str) -> bool:
    """
    Checks, whether area input is a continent
    """
    is_country = (area in countries_gdf["name"].values
                  or area.upper() in countries_gdf["ISO_A3"].values)
    return is_country

def is_continent(area: str) -> bool:
    """
    Checks, whether area input is a continent
    """
    is_continent = area in continents_gdf["CONTINENT_NORM"].values
    return is_continent

def is_world(area: str) -> bool:
    """
    Checks, whether area input is equal to 'world'
    """
    return area == "world"
    
# get geometry for either continent or country
def get_geometry(area: str) -> gpd.GeoDataFrame | None:
    """
    Gets a GeoDataFrame with the geometry of the input area. For 'world', no geometry is needed.
    """
    if type(area) == list:
        geom = None
        return geom
    area = normalise(area)
    if is_country(area):
        geom = countries_gdf[(countries_gdf["name"] == area) |
                             (countries_gdf["ISO_A3"] == area.upper())]
    if is_continent(area):
        geom = continents_gdf[continents_gdf["CONTINENT_NORM"] == area]
    if is_world(area):
        geom = None
    return geom
    

# return list of 4 coordinates for input country
def get_area_coord(area: str) -> list:
    """
    Gets the bounding box coordinates for any valid user input.
    """
    # normalise input using ALIASES dictionary
    area = normalise(area)

    # check validity of area input
    if not is_continent(area) and not is_country(area):
        raise ValueError("Area: Invalid country code, country name or continent.")
    
    # retrieve bbox for continets if input valid
    if is_continent(area):
        continent_gdf = get_geometry(area)
        minx, miny, maxx, maxy = continent_gdf.geometry.total_bounds
        bbox = [minx, miny, maxx, maxy]

    # retrieve bbox for countries if input valid
    else:
        country_gdf = get_geometry(area)
        minx, miny, maxx, maxy = country_gdf.geometry.total_bounds
        bbox = [minx, miny, maxx, maxy]
    print(f"Area name normalised: {area}")
    return [area, ",".join(map(str, bbox))]

# clip data points to country/continent extent
def clip(output_gdf: gpd.GeoDataFrame, area: str) -> gpd.GeoDataFrame:
    """
    Clips the gdf output from the API to the user input area geometry
    """
    area = normalise(area)
    if is_country(area):
        clipped_gdf = gpd.clip(output_gdf, countries_gdf[countries_gdf["name"] == area])
    if is_continent(area):
        clipped_gdf = gpd.clip(output_gdf, continents_gdf[continents_gdf["CONTINENT_NORM"] == area])
    return clipped_gdf

# ----------------------------------------
# Helper function for converting df to gdf
# ----------------------------------------
def df_to_gdf(df: pd.DataFrame) -> gpd.GeoDataFrame:
    """
    Converts the pd.DataFrame output from API query to gpd.GeoDataFrame.
    """
    gdf = gpd.GeoDataFrame(
        data=df,
        geometry=gpd.points_from_xy(df.longitude, df.latitude),
        crs="EPSG:4326"
    )
    return gdf

# ------------
# API Function
# ------------
def area_api_query(sensor: str, area: list|str = 'world', date: str | None = None, n_days: int = 1) -> WildFireQuery:
    """
    Query the NASA FIRMS API for area data for a given sensor and date. If date is not provided, it will return the most recent data.

    Choose one of these sensors:
        'LANDSAT_NRT',
        'MODIS_NRT',
        'MODIS_SP',
        'VIIRS_NOAA20_NRT',
        'VIIRS_NOAA20_SP',
        'VIIRS_NOAA21_NRT',
        'VIIRS_SNPP_NRT',
        'VIIRS_SNPP_SP'

    Parameters:
    -----------
    sensor : str
        Must be from the sensors list.

    area : list or str, optional
        The area for which to query data, 3 options:
        - 'world' for global coverage
        - Country name or ISO 3166-1 alpha-3 country codes (e.g. "Algeria" or "DZA")
        - Continent name (e.g. "North America")
        - Coordinate bounding box with format [west,south,east,north] (e.g. [-180,-90,180,90])

    date : str, optional
        Date in the format 'YYYY-MM-DD'. If not provided, the most recent data will be returned.

    n_days: int, optional
        Number of days that the data should cover. Starting from the stated date
        Max = 5
        Default value = 1

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the area data for the specified sensor and date.
    """
    # sensor
    # ------
    if check_sensor(sensor) == False:
        raise ValueError("Sensor: Invalid sensor name, check list of available sensors.")
    print(f"Sensor input: {sensor}")
    
    # area
    # ----
    print(f"Area input: {area}")
    if type(area) == str:
        if is_world(area):
            area_norm = 'world'
            extent= area
        else:
            area_list = get_area_coord(area)
            area_norm = area_list[0]
            extent = area_list[1]
    elif type(area) == list:
        check_area_list(area)
        extent = ",".join(map(str, area)) # converts input list to str for url use
        area_norm = extent
    else:
        raise ValueError("Area: Anvalid input type. Must be either str or list")
    print(f"Area bbox: {extent}")

    # n_days
    # ------
    check_n_days(n_days)
    n_days = str(n_days)
    print(f"N_days input: {n_days}")

    # date
    # ----
    availability_all_df = get_availability_all(api_key)
    max_date = availability_all_df.loc[
            availability_all_df["data_id"] == sensor,
            "max_date"
        ].iloc[0]
    min_date = availability_all_df.loc[
            availability_all_df["data_id"] == sensor,
            "min_date"
        ].iloc[0]
    max_date_time = datetime.strptime(max_date, "%Y-%m-%d")
    min_date_time = datetime.strptime(min_date, "%Y-%m-%d")
    check_date_str(date, sensor, max_date_time, min_date_time)

    if date is None:
        date = max_date_time - timedelta(days=(int(n_days)-1))
        date_str = date.strftime("%Y-%m-%d")
    if type(date) == str:
        date_str = date
    print(f"Date input: {date_str}")
    print("All input correct")

    # API query
    # ---------
    area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + api_key + '/' + sensor + '/' + extent + '/' + n_days + '/' + date_str
    print(f"URL: {area_url}")
    df = pd.read_csv(area_url)

    # Convert to gpd.GeoDataFrame
    unclipped_gdf = df_to_gdf(df)

    # clip to area boundaries
    if is_world(area):
        output_gdf = unclipped_gdf
    elif type(area) == list:
        output_gdf = unclipped_gdf
    else:
        output_gdf = clip(unclipped_gdf, area)
    print(f"Shape (rows, columns): {output_gdf.shape}")

    # instantiate WildFireQuery object
    output = WildFireQuery(
        data = output_gdf,
        sensor = sensor,
        area = area_norm,
        extent = extent,
        date = date_str,
        n_days = n_days,
        geometry= get_geometry(area)
    )
    print (f'Our current transaction count is {get_transaction_count()}/5000')
    return output

### 1.3. Getting the data
Let us check, whether this function actually works:

In [ ]:
wf = area_api_query(
    'VIIRS_SNPP_SP',
    n_days = 5,
    area='world',
    date='2020-01-01'
    )
display(wf.data.head(1))

There is a case where some of the data is not available (e.g. Australia Bush Fires 2019/2020: Bad API request if area is only australia, but world works). For these cases, set `area = 'world'` in the function call above, `area = `and your desired area and `execute = True` below. This uses the newly defined area and clips the world-data to the smaller geometry, which lets you display more points in your desired area. (This does not change the wf attributes, it is just meant to circumvent the API problem quickly):

In [ ]:
execute = True
area = 'australia'

if execute:
    wf.data = clip(wf.data, area)
    wf.geometry = get_geometry(area)

#### 1.4. Fail-Save and `MAX_ROWS`
This part simply adds a safety mechanism that stops the notebook if the dataset fetched is too large and could cause VSCode to crash when trying to map it (happened to me with 190'000 fire points). Or, of course, if it has no entries :)

In [ ]:
MAX_ROWS = 10000

if len(wf.data) > MAX_ROWS:
    wf.sampled = True
    warnings.warn(
        f"Dataset too large for individual point plotting ({len(wf.data)} rows). "
        f"A random subset with n={MAX_ROWS} is displayed instead",
        UserWarning
    )
if len(wf.data) == 0:
    raise ValueError("Empty dataset: nothing to visualise here. Try other parameters.")

***
## 2. Cleaning the dataframe
Before trying to visualise all the interesting data contained in the dataset, we must clean it first and prepare it for future steps. We can get an overview of the dataset by looking at the columns and what type of data they contain.



In [ ]:
overview(wf.data)

Based on these insights, we can now decide how to clean and change the variables and entries:
- change the acquisition date and acquisition time to a valid datetime format.
- check plausibility of values
- drop the brighness columns since we do not need them in this project
- drop low confidence entries

It seems like the API delivers an already clean data set which is why no more steps are needed.
### 2.1. Cleaning Function
At the moment, the time and date are stored in separate columns as `str`. The following step will convert it into a single column containing gpd.datetime objects. Afterwards, we can drop the now useless `acq_time` and `acq_date` columns.  

Additionally we check for each variable, whether the values are plausible. This includes:
- remove rows with negative FRP values
- e.g. latitude above 90 degrees
- dropping rows with confidence below 30% or scored as 'l'
- removes potentially duplicate entries (with same lat/lon and time)


In [ ]:
def clean(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Cleans a raw wildfire GeoDataFrame by removing invalid rows.
    Drops rows silently and prints a summary of what was removed.
    - Converts acq_time and acq_date to datetime
    - Checks coordinate plausibility
    - Drops rows with frp < 0
    - Normalises frp to MW per km^2
    - Removes all brightness columns
    - Checks confidence plausibility
    - Removes low confidence entries
    - Checks Day/Night
    - Drops duplicate rows

    Parameters:
    -----------
    gdf : gpd.GeoDataFrame
        Raw GeoDataFrame from FIRMS API query

    Returns:
    --------
    gpd.GeoDataFrame
        Cleaned GeoDataFrame
    """
    original_len = len(gdf)
    dropped = {}

    # dropping rows with NaN for required columns
    # -------------------------------------------
    required_cols = ['latitude', 'longitude', 'scan', 'track', 'confidence', 'acq_time', 'acq_date']
    mask = gdf[required_cols].notna().all(axis=1)
    dropped["missing_required"] = (~mask).sum()
    gdf = gdf[mask]

    # Datetime
    # --------
    if "acq_time" in gdf.columns:
        gdf["datetime"] = pd.to_datetime(
            gdf["acq_date"] + " " + gdf["acq_time"].astype(str).str.zfill(4),
            format="%Y-%m-%d %H%M",
            utc=True
        )
        gdf = gdf.drop(columns=["acq_time", "acq_date"])

    # Coordinates
    # -----------
    mask = (
        gdf["latitude"].between(-90, 90) &
        gdf["longitude"].between(-180, 180)
    )
    dropped["invalid_coordinates"] = (~mask).sum()
    gdf = gdf[mask]

    # FRP cleaning
    # ------------
    if "frp" in gdf.columns:
        mask = gdf["frp"] > 0
        dropped["invalid_frp"] = (~mask).sum()
        gdf = gdf[mask]
    else:
        dropped["invalid_frp"] = 0
        gdf["frp"] = 0

    # FRP normalisation
    # -----------------
    if "frp" in gdf.columns:
        gdf["frp_density"] = (gdf["frp"] / (gdf["scan"] * gdf["track"])).round(1) # MW per km^2

    # Brightness temperatures
    # -----------------------
    bt_checks = ["brightness", "bright_t31", "bright_ti4", "bright_ti5"]

    for col in bt_checks:
        if col in gdf.columns:
            gdf = gdf.drop(columns=[col])

    # Confidence
    # ----------
    # MODIS
    if pd.api.types.is_numeric_dtype(gdf["confidence"]):
        # remove invalid confidence
        mask_valid = gdf["confidence"].between(0, 100)
        dropped["invalid_confidence"] = (~mask_valid).sum()
        gdf = gdf[mask_valid]
        # remove unconfident entries: > 30%
        mask_likely = gdf["confidence"] > 30
        dropped["low_confidence"] = (~mask_likely).sum()
        gdf = gdf[mask_likely]
    # VIIRS
    else:
        # convert Landsat confidence to lowercase
        gdf["confidence"] = gdf["confidence"].str.lower()
        # remove unconfident entries: "l" (= low)
        dropped["invalid_confidence"] = 0
        mask_likely = gdf["confidence"].isin(["n", "h"])
        dropped["low_confidence"] = (~mask_likely).sum()
        gdf = gdf[mask_likely]

    # Day/Night flag
    # --------------
    mask = gdf["daynight"].isin(["D", "N"])
    dropped["invalid_daynight"] = (~mask).sum()
    gdf = gdf[mask]

    # Type
    # ----
    if 'type' in gdf.columns:
        mask = gdf['type'].isin([0,1,2,3])
        dropped['invalid_type'] = (~mask).sum()
        gdf = gdf[mask]

    # Duplicates
    # ----------
    # also removes duplicates accross different satellites
    duplicate_mask = gdf.duplicated(subset=["latitude", "longitude", "datetime"])
    dropped["duplicates"] = duplicate_mask.sum()
    gdf = gdf[~duplicate_mask]

    # Summary
    # -------
    total_dropped = original_len - len(gdf)
    print(f"=== Cleaning Summary ===")
    print(f"Missing required fields : {dropped['missing_required']}")
    print(f"Invalid coordinates     : {dropped['invalid_coordinates']}")
    if 'frp' in gdf.columns:
        print(f"Invalid FRP             : {dropped['invalid_frp']}")
    print(f"Invalid confidence      : {dropped['invalid_confidence']}")
    print(f"Low confidence          : {dropped['low_confidence']}")
    print(f"Invalid daynight        : {dropped['invalid_daynight']}")
    if 'type' in gdf.columns:
        print(f"Invalid type            : {dropped['invalid_type']}")
    print(f"Duplicates              : {dropped['duplicates']}")
    print(f"-------------------------")
    print(f"Rows removed: {total_dropped} of {original_len}")
    print(f"Rows remaining: {len(gdf)}")

    return gdf.reset_index(drop=True)

Now that we have defined the function, we can use it on the `.data` attribute to create a new attribute called `.cleaned`. The summary also shows what the cleaning function did to the dataset, and most importantly the dimensions of the dataset afterwards, so we can check, whether we are below `MAX_ROWS`.

In [ ]:
wf.cleaned = clean(wf.data)

## 3. Interactive Map
Now we can tackle the heart of the project: the visualisation. The plan is to create a singular interactive map with folium and then let the user toggle each layer to their liking.
### 3.1. Clustering
If we want to find especially large fires, we need to cluster single fire pixels together. The best approach for this is to use DBSCAN, since it handles noise well and we don't have to predefine the number of clusters. The two parameters we can define are `min_samples` (the minimum number of fire pixels in one cluster, all other points are not assigned to any cluster), and `eps` (epsilon: maximum distance from nearest neighbour in a cluster). In our case, an epsilon lower than the pixel size of the sensor would result in practically no clusters, since the fire pixels are by design of the sensor at least one pixel size apart. But if we make epsilon too large, we could risk grouping several fires into one.  

Another thing to note about this clustering approach is that we assume a perfectly sperical earth, which results in approximations instead of accurate calculations for radians. We are using the volumetric mean radius of the earth, meaning it maps a larger distance at the equator and a smaller distance at the poles. Nevertheless, we assume the errors to be negligible based on the following calculations:
- *Equator*:  
    for epsilon_km = **1.00000**: true distance = (1 / Volumetric mean radius) * true equatorial radius = (1 / 6371.0088) * 6378.137 = **1.00112 km (+1.12m)** which is a **0.112% error**.
- *Poles*:  
    for epsilon_km = **1.00000**: true distance = (1 / Volumetric mean radius) * true polar radius = (1 / 6371.0088) * 6356.752 = **0.99776 km (-2.24m)** which is a **0.224% error**.  

In the clustering function below, we:
- cluster the points using DBSCAN
- add the clustering ID we obtained from the previous step to our GDF which still includes all points.
- We group this GDF by cluster ID and dissolve.
- The, we calculate the centroid for each dissolved cluster. We are now left with a GDF of only centroids.
- During the entire process, we use a special feature engineering function, which transfers the variables from the old GDF to the new, cluster-based GDF.

In [ ]:
# Helper functions
# ----------------
def cluster_feature_engineering(gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Generates a 'stats' DF which contains all the necessary attributes of the fire clusters.
    """
    # fixed feature engineering
    agg_kwargs = {'pixel_count': ('geometry', 'count'),
                'first_pixel': ('datetime', 'min'),
                'last_pixel': ('datetime', 'max'),
                'time_mean': ('datetime', 'mean')}
    
    # preserve 'type' column if available
    if 'type' in gdf.columns:
        agg_kwargs['type'] = ('type', 'first')
    
    # engineer frp density if available
    if 'frp_density' in gdf.columns:
        agg_kwargs['frp_sum'] = ('frp', 'sum')
        agg_kwargs['frp_mean'] = ('frp', 'mean')

    # uses the specific agg_kwargs create stats DataFrame for clusters (groupy)
    stats = gdf.groupby('cluster').agg(**agg_kwargs)

    # and rounds the values for readability
    round_cols = []
    if 'frp_density' in gdf.columns:
        round_cols.append('frp_mean')
        round_cols.append('frp_sum')
    if round_cols:
        stats[round_cols] = stats[round_cols].round(1)
    
    return stats

def convert_type(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    # convert type from ordinal numbers to readable names
    types_dict = {
        0: 'vegetation fire',
        1: 'active volcano',
        2: 'other (land)',
        3: 'offshore'
    }
    if 'type' in gdf.columns:
        gdf['type_str'] = gdf['type'].apply(lambda x: types_dict[x])

    return gdf

# clustering function
# -------------------
def cluster_wf(wf: WildFireQuery, min_samples=2) -> gpd.GeoDataFrame:
    """
    Clusters the fire pixels into groups representing individual fires using DBSCAN

    Parameters:
    -----------
    wf: WildFireQuery, Cleaned GDF from FIRMS API query
    min_samples: int, min points within eps to be considered core point

    Returns:
    --------
    clustered_gdf: gpd.GeoDataFrame, Clustered GDF GeoDataFrame
    """
    # Setting Parameters
    # ------------------
    # volumetric mean readius of earth
    # used, because Haversine uses radians instead of degrees
    KM_PER_RADIAN = 6371.0088 
    epsilon_km = wf.dbscan_epsilon

    # 2 different approaches, dependon on whether 'type' column is present
    # --------------------------------------------------------------------
    db_params = DBSCAN(
        # min distance to nearest neighbor to be included in the cluster
        eps=epsilon_km / KM_PER_RADIAN,
        # min points within eps to be considered core point (essentially sets min cluster size)
        min_samples=min_samples,
        # required when using haversine
        algorithm='ball_tree',
        # handles lat, lon coordinates, as compared to using metric and having to reproject twice
        metric='haversine'
        )
    
    # cluster by 'type'
    if 'type' in wf.cleaned.columns and wf.cleaned['type'].notna().any(): 
        all_clusters = []
        offset = 0
        
        # splits the dataset into subsets by fire type and iterates over each type
        for fire_type, group in wf.cleaned.groupby('type'): 
            # extract lon/lat for each fire pixel and convert to radians
            coords_rad = np.radians(
                np.column_stack([group.geometry.y, group.geometry.x])
            )

            # assuming spherical earth
            db = db_params.fit(coords_rad)
            
            labels = db.labels_.copy()
            # add offset to IDs to avoid ID collisions
            labels[labels >= 0] += offset  
            offset = labels.max() + 1 if labels.max() >= 0 else offset
            
            all_clusters.append(pd.Series(labels, index=group.index))
        
        wf.cleaned['cluster'] = pd.concat(all_clusters)

        clustered = wf.cleaned[wf.cleaned['cluster'] >= 0].copy()

    # disregard 'type'
    else: # same procedure
        coords_rad = np.radians(
            np.column_stack([wf.cleaned.geometry.y, wf.cleaned.geometry.x])
        )
        db = db_params.fit(coords_rad)
        wf.cleaned['cluster'] = db.labels_

        clustered = wf.cleaned[wf.cleaned['cluster'] >= 0].copy()

    # feature preservation and engineering for dissolve
    # -------------------------------------------------
    stats = cluster_feature_engineering(clustered)

    # dissolve (going from points to clusters)
    # ----------------------------------------
    # one centroid point per cluster
    centroids = clustered.dissolve(by='cluster').centroid.rename('geometry')

    # feature engineering in clustered GDF
    # ------------------------------------
    # add stats DF to centroids GDF
    cluster_gdf = gpd.GeoDataFrame(stats, geometry=centroids, crs=wf.cleaned.crs)
    # calculate timespan of fire
    cluster_gdf["time_span"] = cluster_gdf["last_pixel"] - cluster_gdf["first_pixel"]
    # convert type to readable fire types
    cluster_gdf = convert_type(cluster_gdf)
    # sort by size
    cluster_gdf = cluster_gdf.sort_values("pixel_count", ascending=False)

    return cluster_gdf

In [ ]:
wf.clustered = cluster_wf(wf)
overview(wf.clustered)

The `UserWarning` can be ignored. There is no significant geographic deviation when using a non projected crs.

### 3.2. Severity score vor VIIRS SP
We also want to give the user an insight into where severe fires are. This is done by using some of the variables, which come with the FIRMS API data, to calculate a severity score. The variables used are:
- Total FRP over fire area: combines size and intensity
- Mean FRP: research ([Urfali & Eymen, 2025](https://doi.org/10.3390/fire8080308)) shows that higher mean FRP correlates with higher severity. By averaging all pixel FRP values, we also get a comparable value between clusters.
- `pixel_count` corrected with `self.pixel_area`
- brightness is not used, since it is a direct construction parameter for frp, which means they are correlated already.  

Since these variables strongly depend on the type of instrument (MODIS or VIIRS) that was used to capture data, the severity score in this project is only valid for VIIRS SP data.

Regarding Pixel count, there are a few more steps involved to get to an approximation of the fire's area:
1. By multiplying the pixel count with the area that each pixel represents, we can get a first estimate. Often times, this is multiple times larger than the real area.
2. By measuring some of the fire areas by hand, we can calculate instrument-dependent scalar by which we need to shrink the first estimate (Modis: 2/7, VIIRS: 2/3).
#### 3.2.1. Normalisation
We first normalise the input values to a range of 0 to 1.
For normalisation, we have looked at the 2023 canadian wildfires as a reference for very severe wild fires:
- highest FRP sum: MW
- highest FRP mean:  MW
- largest cluster: km^2

We take the maximum value for 5 different timeframes during the 2023 Canadian wildfires use the max value of those as our normalisation parameter: 

VIIRS:
| Date      | 13.5.     | 17.5.     | 23.5.     | 26.5.     | 10.6.     | MAX       |
|-----------|-----------|-----------|-----------|-----------|-----------|-----------|
| FRP sum   | 100067.9  | 70221.4   | 88894.2   | 119387.4  | 42020.5   | 119387.4  |
| FRP mean  | 227.7     | 158.9     | 276.8     | 276.8     | 122.5     | 276.8     |
| Size      | 892.6     | 543.6     | 321.7     | 548.4     | 211.9     | 892.6     |

#### 3.2.2. Weights
The penultimate step is to distribute the weights for the score calculation:
- FRP sum: 0.1
- FRP mean: 0.5
- Cluster size: 0.4  

The weights are split equally between variables driven by size (FRP sum and cluster size) and FRP mean (not strongly driven by size)  
FRP sum has such a low weight, because it is somewhat correlated to cluster size and we already include a variable connected to FRP which is strongly weighted.

#### 3.2.3. Severity Score calculation
The severity score is calculated as follows:
$$\text{severity\_score} = w_1 \cdot \overline{frp}_{sum,norm} + w_2 \cdot \overline{frp}_{mean,norm} + w_3 \cdot \overline{size}_{norm}$$
To get a more realistic score, we can use a log-transformation. This spreads out lower and crushes higher score values, which is much more representative of real life wild fire severity. As with many natural phenomena, FRP values for wild fires are heavily right skewed. A linear score inherits this disproportionate distribution, while a logarithmic scale accounts for this ([Wooster et al., 2005](https://doi.org/10.1029/2005JD006318)).

By applying the following log-transformation, we get a score on the scale between 1 and 10 (while we still call this a severity score, in the code it is called severity_class for distinction):
$$\text{severity\_class} = \frac{\ln(2 + 9 \cdot \text{severity\_score})}{\ln(11)} \cdot 9 + 1$$
We basically transform on the scale between 0 to 1, multiply by 9 to get a score between 0 and 9 and then add 1 (so we don't get score values below 1).  

This is still not enough, though, since the wildfire data now only maps to scores between 3.5 and roughly 7. Assuming that the Canadian Bush Fires of 2023 deserve one of the higher severity scores and single pixel fires a score of about 1, we should adjust the scale. We can allow ourselves such a manual change, since the score's scale is rather arbitrary, and values starting at 1 are arguably more intuitive than values between 3.5 to 7.5.
$$\text{severity\_class} = \text{clip}\left(\frac{\text{severity\_class} - 3.5}{4} \cdot 9 + 1, 1, 10\right)$$

In [ ]:
def severity_score(wf: WildFireQuery, weights: list) -> gpd.GeoDataFrame:
    """ 
    Calculates a severity score and class for fire pixel clusters and adds it to gdf.  
    It uses:
    - FRP sum
    - FRP mean
    - cluster size  
    with user specified weights

    Parameters:
    -----------
    wf : WildFireQuery, entire class object containing the clustered gdf as attribute
    weights: list, List with 3 integer values which specify the weights in the order [frp sum, frp mean, cluster size]

    Returns:
    --------
    gpd.GeoDataFrame: The severity score/class is added as a column.
    """
    # raise error if weights do not sum to 1
    if round(sum(weights), 4) != 1.0:
        raise ValueError(f"Weights must sum to 1, got {sum(weights)}")
    
    s_gdf = wf.clustered.copy()
    FRP_SUM_VIIRS = 119387   # MW
    FRP_MEAN_VIIRS = 277    # MW
    CLUSTER_SIZE = 893      # km2
    
    # calculate fire cluster duration for normalisation later (rounds to higher integer, at least 1)
    s_gdf["time_span_days"] = np.ceil(s_gdf["time_span"].dt.total_seconds() / 86400).clip(1, 5).round(1)

    if wf.instrument == 'LANDSAT':
        s_gdf["cluster_size_approx"] = ((s_gdf["pixel_count"] * wf.pixel_area)).round(3)
        s_gdf["severity_class"] = 'unknown'
        s_gdf["frp_mean"] = 'unknown'
        s_gdf["frp_sum"] = 'unknown'
        
    elif wf.instrument == 'VIIRS':
        # frp sum
        s_gdf["frp_sum_day"] = (s_gdf["frp_sum"] / s_gdf["time_span_days"]).round(1)
        s_gdf["frp_sum_day_norm"] = (s_gdf["frp_sum_day"] / FRP_SUM_VIIRS).clip(0, 1)
        # frp mean
        s_gdf["frp_mean_norm"] = (s_gdf["frp_mean"] / FRP_MEAN_VIIRS).clip(0, 1)
        # cluster size in km2
        s_gdf["cluster_size_approx"] = ((s_gdf["pixel_count"] * wf.pixel_area) / 3.5).round(3)

        # normalise
        s_gdf["cluster_size_approx_norm"] = (s_gdf["cluster_size_approx"] / CLUSTER_SIZE).clip(0, 1)

        s_gdf["severity_score"] = (s_gdf["frp_sum_day_norm"] * weights[0] +
                                s_gdf["frp_mean_norm"] * weights[1] +
                                s_gdf["cluster_size_approx_norm"] * weights[2])
        
        # creates log scale between 0 and 1 and then multiplies by 9 -> range 0 to 10,
        # and adds 1 to get to scale between 1 and 10
        s_gdf["severity_class"] = (
            np.log1p(s_gdf["severity_score"] * 9 + 1) / np.log1p(10) * 9 + 1
        ).round(1)

        # since 3.5 seems to be the lowest possible value, we stretch it to 1,
        # 6.7 becomes 10 and it now is an almost open ended scale (technically
        # max is 20, which was 10 before stretching)
        s_gdf["severity_class"] = (
            (s_gdf["severity_class"] - 3.5) / (6.7 - 3.6) * 9 + 1
        ).clip(lower=1).round(1)

    elif wf.instrument == 'MODIS':
        # cluster size in km2 (Factor 3.5: )
        s_gdf["cluster_size_approx"] = ((s_gdf["pixel_count"] * wf.pixel_area) / 1.5).round(3)
        s_gdf["severity_class"] = 'unknown'

    return s_gdf

In [ ]:
weights = [0.4, 0.1, 0.5] # [frp sum, frp mean, cluster size]
wf.clustered = severity_score(wf, weights)

### 3.3. Mapping
Finally, we can visualise what we just fetched, cleaned, restructured, clustered and calculated. The result is a single folium map with layers that the user can toggle. This allows for interactive exploration of the wildfire data in the user's desired area.  

The map layers are created in the order they are stated in the docstring:
- Area outline
- heatmap
- Fires (clustered pixels)
- Severity Score (Only VIIRS_xxx_SP sensors)
- Fire pixels by Fire Radiative Power in megawatts (VIIRS and MODIS)
- Fire pixels by detection time (VIIRS and MODIS)
- Volcanoes (Only xxx_xxx_SP sensors)
- Offshore fires (Only xxx_xxx_SP sensors)
- Other landbased fire sources (Only xxx_xxx_SP sensors)  

Depending on the sensor and instrument from which the data comes, different layers are available.
- LANDSAT provides the least layers, due to the lac of FRP values, which are the base of most of the more advanced wild fire analysis done in this project.
- MODIS provides everything VIIRS does, except for the severity score.
- VIIRS is the most versatile sensor when it comes to map layers.
- Some instrument groupd are split between NRT (near real time) and SP (standard processing).
    - SP provides a variable `type` which means we can also display the 4 different types of fires (vegetation, other land source, volcanoes, off shore). The severity score is also only available in VIIRS_xxx_SP, not NRT.
    - NRT is less advanced: without the `type` variable it can neither show type (obviously) nor severity score

In [ ]:
def map_wf(wf: WildFireQuery, save: bool = True) -> fm.Map:
    """
    map_wf creates a folium map with wildfire data from FIRMS API

    Map layers:
    - Area outline
    - heatmap
    - Fires (clustered pixels)
    - Severity Score (Only VIIRS_xxx_SP sensors)
    - Fire pixels by Fire Radiative Power (VIIRS and MODIS)
    - Fire pixels by detection time (VIIRS and MODIS)
    - Volcanoes (Only xxx_xxx_SP sensors)
    - Offshore fires (Only xxx_xxx_SP sensors)
    - Other landbased fire sources (Only xxx_xxx_SP sensors)

    Parameters:
    -----------
    wf : WildFireQuery, entire class object which has the cleaned and clustered gdf stored as attributes.
    save: bool, toggle saving map as html and open in browser automatically

    Returns:
    --------
    m : fm.Map, a folium map with all the layers above for interactively exploring the wild fire data.
    file : if save == True, saves file to current working directory
    """
    # ---------------------------------
    # prepare clusters gdf for mapping in folium
    # ---------------------------------
    # convert datetime to strings for JSON handling
    cluster_gdf_plot = wf.clustered.copy()
    severity_gdf_plot = wf.clustered.copy()
    if len(cluster_gdf_plot) > MAX_ROWS:
        cluster_gdf_plot = cluster_gdf_plot.sort_values("pixel_count", ascending = False)
        cluster_gdf_plot = cluster_gdf_plot.head(MAX_ROWS)
        wf.sampled = True
        severity_gdf_plot = severity_gdf_plot.sort_values("severity_class", ascending = False)
        severity_gdf_plot = severity_gdf_plot.head(MAX_ROWS)
        wf.sampled = True
    # datetimes
    cluster_gdf_plot["first_pixel"] = cluster_gdf_plot["first_pixel"].dt.strftime("%Y-%m-%d %H:%M UTC")
    cluster_gdf_plot["last_pixel"] = cluster_gdf_plot["last_pixel"].dt.strftime("%Y-%m-%d %H:%M UTC")
    cluster_gdf_plot["time_mean"] = cluster_gdf_plot["time_mean"].dt.strftime("%Y-%m-%d %H:%M UTC")
    cluster_gdf_plot["time_span"] = cluster_gdf_plot["time_span"].apply(
        lambda x: f"{int(x.total_seconds() // 3600)}h {int((x.total_seconds() % 3600) // 60)}m" # lambda just used to construct the string and pass x
    )

    # add column to use later for tooltip: what kind of object am I looking at?
    cluster_gdf_plot["display_type"] = "Fire Cluster"
    severity_gdf_plot["display_type"] = "Fire Cluster"

    # ---------------------------------
    # prepare point gdf for mapping in folium
    # ---------------------------------
    gdf_plot = wf.cleaned.copy()
    if len(gdf_plot) > MAX_ROWS:
        gdf_plot = gdf_plot.sample(MAX_ROWS)
        wf.sampled = True
    gdf_plot['display_type'] = 'Fire Pixel' # add column to use later for tooltip: what kind of object am I looking at?
    gdf_plot["datetime_num"] = gdf_plot["datetime"].astype("int64") // 1e9 # convert nanoseconds to seconds
    gdf_plot["datetime"] = gdf_plot["datetime"].dt.strftime("%Y-%m-%d %H:%M UTC")

    # ---------------------------------
    # initiate the folium map canvas
    # ---------------------------------
    center = wf.get_center()
    zoom_start = wf.get_zoom_level()
    m = fm.Map(
        location=center,
        zoom_start=zoom_start,
        min_zoom = zoom_start,
        control_scale=True,
        tiles= "CartoDB DarkMatter")
    
    # ---------------------------------
    # Add Data Information
    # ---------------------------------
    title_html = f"""
    <div style="
        position: fixed;
        top: 10px;
        right: 10px;
        z-index: 1000;
        background-color: rgba(0,0,0,0.6);
        color: white;
        padding: 10px 15px;
        border-radius: 5px;
        font-family: Arial;
        font-size: 13px;
    ">
        <b>Wildfire Detections</b><br>
        Sensor: {wf.sensor}<br>
        Date: {wf.date}<br>
        Area: {wf.area_display}<br>
        Days: {wf.n_days}<br>
        Sampled: {wf.sampled}
    </div>
    """

    m.get_root().html.add_child(fm.Element(title_html))

    # ---------------------------------
    # area outline
    # ---------------------------------
    if wf.geometry is not None:
        outline_group = fm.FeatureGroup(name="Area outline", show=True)
        fm.GeoJson(wf.geometry,
                style_function=lambda x: {
                    "color": "white",
                    "weight": 1,
                    "opacity": 0.7,
                    "fillOpacity": 0
                    }
                ).add_to(outline_group)
        outline_group.add_to(m)

    # --------------------------------------------------------------------------------------
    # Heatmap
    # ---------------------------------
    heat_data = wf.cleaned[["latitude", "longitude", "frp_density"]].values.tolist()
    heat_group = fm.FeatureGroup(name="Heatmap", show=True)
    plugins.HeatMap(
        heat_data,
        min_opacity=0.4,
        radius=8,
        blur=6,
        max_zoom=10
        ).add_to(heat_group)
    heat_group.add_to(m)

    # ---------------------------------
    # prepare helpers for clusters
    # ---------------------------------
    # style function
    max_pixel_count = cluster_gdf_plot['pixel_count'].max()
    def make_style_fn(color, min_opacity):
        def style_fn(feature):
            pixel_count = feature['properties']['pixel_count']
            return {
                'radius': max(3, pixel_count ** 0.7),
                'color': color,
                'weight': 1,
                'opacity': max(min_opacity, pixel_count / max_pixel_count)
            }
        return style_fn
    
    # tooltip
    def make_cluster_tooltip():
        return fm.GeoJsonTooltip(
            fields=["display_type", "pixel_count", "cluster_size_approx", "frp_mean", "frp_sum", "time_span"],
            aliases=["Object:", "# pixels in cluster", "~ cluster size [km^2]", "mean FRP [MW]", "FRP sum [MW]", "Time Span:"]
            )

    # ---------------------------------
    # fire clusters
    # ---------------------------------
    fire_clusters = fm.FeatureGroup(name="Fires", show=False)
    fm.GeoJson(
        cluster_gdf_plot[["geometry", "display_type", "pixel_count", "frp_sum", "frp_mean", "cluster_size_approx", "time_span"]],
        marker=fm.CircleMarker(
            fill=True,
            fill_opacity=0,
        ),
        tooltip=make_cluster_tooltip(),
        style_function=make_style_fn('cyan', 0.5),
    ).add_to(fire_clusters)
    fire_clusters.add_to(m)
    # ---------------------------------
    # severity score
    # ---------------------------------
    if wf.sensor in ['VIIRS_NOAA20_SP', 'VIIRS_SNPP_SP']:
        # define rendering order
        severity_gdf_plot = severity_gdf_plot.sort_values("severity_class", ascending=True)

        # create color ramp
        severity_colors = [
            "#3f007d", "#54278f", "#6a51a3", "#807dba", "#9e9ac8",
            "#bcbddc", "#dadaeb", "#f2f0f7", "#f8f4ff", "#ffffff"
        ]
        colormap_severity = cm.StepColormap(
                    colors = severity_colors,
                    vmin=1,
                    vmax=10
                    )
        colormap_severity.caption = "WFSS"

        # plot points
        severity_clusters = fm.FeatureGroup(name="WFSS", show=False)
        fm.GeoJson(
            severity_gdf_plot[["geometry", "display_type", "severity_class"]],
            marker=fm.CircleMarker(
                fill=True,
                fill_opacity=0.02,
                weight=1,
            ),
            tooltip=fm.GeoJsonTooltip(
                fields=["display_type", "severity_class"],
                aliases=["Object:", "WFSS"]
                ),
            style_function=lambda feature: {
                        "color": colormap_severity(feature["properties"]["severity_class"]),
                        "fillColor": colormap_severity(feature["properties"]["severity_class"]),
                        "radius": max(1, feature["properties"]["severity_class"] ** 1.9),
                    },
        ).add_to(severity_clusters)
        severity_clusters.add_to(m)

        # legend
        severity_legend_html = """
            <div style="
                position: fixed;
                bottom: 310px;
                right: 10px;
                z-index: 1000;
                background-color: rgba(0,0,0,0.7);
                color: white;
                padding: 10px 15px;
                border-radius: 5px;
                font-family: Arial;
                font-size: 12px;
            ">
                <b>WFSS</b><br>
            """
        for i, color in enumerate(severity_colors):
            label = "> 10" if i == len(severity_colors) - 1 else i + 1
            severity_legend_html += f"""
                <div style="display:flex; align-items:center; margin-top:4px">
                    <div style="background:{color}; width:20px; height:20px; margin-right:8px; border-radius:3px; border:1px solid rgba(255,255,255,0.2)"></div>
                    {label}
                </div>
                """
        severity_legend_html += "</div>"
        m.get_root().html.add_child(fm.Element(severity_legend_html))


    # --------------------------------------------------------------------------------------
    # prepare helpers for point layers
    # ---------------------------------
    def make_point_tooltip():
        return fm.GeoJsonTooltip(
            fields=["display_type", "frp", "datetime"],
            aliases=["Object:", "FRP [MW]:", "Time:"]
            )

    # ---------------------------------
    # Fire pixels by FRP -> ranked
    # ---------------------------------
    if wf.instrument != 'LANDSAT':
        quantiles = [0, 0.2, 0.4, 0.6, 0.8, 1.0]  # 5 equal classes
        breaks = wf.cleaned["frp"].quantile(quantiles).values

        colormap_frp = cm.StepColormap(
            colors=["#bd0026", "#f03b20", "#fd8d3c", "#fecc5c", "#ffffb2"],
            index=breaks,
            vmin=breaks[0],
            vmax=breaks[-1]
            )
        
        colormap_frp.caption = "FRP [MW]"

        # plot
        frp_points_group2 = fm.FeatureGroup(name="Fire Pixels by FRP", show=False)
        fm.GeoJson(
            gdf_plot[["geometry", "display_type", "frp", "datetime"]].sort_values("frp", ascending=True),
            marker=fm.CircleMarker(radius=2,
                                fill=True,
                                fill_opacity=0.7,
                                weight=0
                                ),
            style_function=lambda feature: {
                "color": colormap_frp(feature["properties"]["frp"]),
                "fillColor": colormap_frp(feature["properties"]["frp"])
            },
            tooltip=make_point_tooltip()
        ).add_to(frp_points_group2)
        frp_points_group2.add_to(m)

        # html legend
        frp_html = """
        <div style="
            position: fixed;
            bottom: 150px;
            right: 10px;
            z-index: 1000;
            background-color: rgba(0,0,0,0.7);
            color: white;
            padding: 10px 15px;
            border-radius: 5px;
            font-family: Arial;
            font-size: 12px;
        ">
            <b>Fire Radiative Ppower [MW]</b><br>
        """

        colors = ["#ffffb2", "#fecc5c", "#fd8d3c", "#f03b20", "#bd0026"]
        for i in range(len(colors)):
            frp_html += f"""
            <div style="display:flex; align-items:center; margin-top:4px">
                <div style="background:{colors[i]}; width:20px; height:20px; margin-right:8px; border-radius:3px"></div>
                {breaks[i]:.2f} – {breaks[i+1]:.2f}
            </div>
            """

        frp_html += "</div>"
        m.get_root().html.add_child(fm.Element(frp_html))

    # ---------------------------------
    # Fire pixels by detection datetime
    # ---------------------------------
    date_min = gdf_plot["datetime_num"].min()
    date_max = gdf_plot["datetime_num"].max()

    colors = cm.linear.YlOrRd_09.colors[::-1]

    colormap_time = cm.LinearColormap(
        colors=colors,
        vmin=date_min,
        vmax=date_max
    )

    # plot
    time_points_group = fm.FeatureGroup(name="Fire Pixels by detection time", show=False)
    fm.GeoJson(
        gdf_plot[["geometry", "display_type", "frp", "datetime", "datetime_num"]],
        marker=fm.CircleMarker(radius=2,
                            fill=True,
                            fill_opacity=0.7,
                            weight=0
                            ),
        style_function=lambda feature:{
            "color": colormap_time(feature["properties"]["datetime_num"]),
            "fillColor": colormap_time(feature["properties"]["datetime_num"])
        },
        tooltip=make_point_tooltip()
    ).add_to(time_points_group)
    time_points_group.add_to(m)

    # html legend
    time_start = wf.cleaned["datetime"].min().strftime("%Y-%m-%d %H:%M UTC")
    time_end = wf.cleaned["datetime"].max().strftime("%Y-%m-%d %H:%M UTC")

    time_legend_html = f"""
    <div style="
        position: fixed;
        bottom: 30px;
        right: 10px;
        z-index: 1000;
        background-color: rgba(0,0,0,0.7);
        color: white;
        padding: 10px 15px;
        border-radius: 5px;
        font-family: Arial;
        font-size: 12px;
    ">
        <b>Detection Time</b><br><br>
        <div style="
            width: 150px;
            height: 15px;
            background: linear-gradient(to right, #ffffb2, #bd0026);
            border-radius: 3px;
            margin-bottom: 4px;
        "></div>
        <div style="display:flex; justify-content:space-between; width:150px">
            <span>{time_start}</span>
            <span>{time_end}</span>
        </div>
    </div>
    """
    m.get_root().html.add_child(fm.Element(time_legend_html))

    # ---------------------------------
    # Volcano pixels
    # ---------------------------------
    if "type" in wf.cleaned.columns:
        volcanoes_points = gdf_plot[gdf_plot["type"] == 1][
            ["geometry", "display_type", "frp", "datetime"]
            ]
        if len(volcanoes_points) < MAX_ROWS and len(volcanoes_points) > 0:
            volcano_pixels_group = fm.FeatureGroup(name="Volcano Pixels", show=False)
            fm.GeoJson(
                volcanoes_points,
                marker=fm.CircleMarker(radius=2,
                                    fill=True,
                                    fill_color="lime",
                                    fill_opacity=1,
                                    weight=0
                                    ),
                tooltip=make_point_tooltip()
            ).add_to(volcano_pixels_group)
            volcano_pixels_group.add_to(m)
        volcanoes_clusters = cluster_gdf_plot[cluster_gdf_plot["type"] == 1][
            ["geometry", "display_type", "pixel_count", "frp_sum", "frp_mean", "cluster_size_approx", "time_span"]
            ]
        if len(volcanoes_clusters) < MAX_ROWS and len(volcanoes_clusters) > 0:
            volcano_clusters_group = fm.FeatureGroup(name="Volcano Clusters", show=False)
            fm.GeoJson(
                volcanoes_clusters,
                marker=fm.CircleMarker(
                    fill=True,
                    fill_opacity=0,
                ),
                tooltip=make_cluster_tooltip(),
                style_function=make_style_fn('lime', 1),
            ).add_to(volcano_clusters_group)
            volcano_clusters_group.add_to(m)

    # ---------------------------------
    # Offshore pixels
    # ---------------------------------
    if "type" in wf.cleaned.columns:
        offshore_points = gdf_plot[gdf_plot["type"] == 3][
            ["geometry", "display_type", "frp", "datetime"]
            ]
        if len(offshore_points) < MAX_ROWS and len(offshore_points) > 0:
            offshore_pixels_group = fm.FeatureGroup(name="Offshore Pixels", show=False)
            fm.GeoJson(
                offshore_points,
                marker=fm.CircleMarker(radius=2,
                                    fill=True,
                                    fill_color="yellow",
                                    fill_opacity=1,
                                    weight=0
                                    ),
                tooltip=make_point_tooltip()
            ).add_to(offshore_pixels_group)
            offshore_pixels_group.add_to(m)
        offshore_clusters = cluster_gdf_plot[cluster_gdf_plot["type"] == 3][
            ["geometry", "display_type", "pixel_count", "frp_sum", "frp_mean", "cluster_size_approx", "time_span"]
            ]
        if len(offshore_clusters) < MAX_ROWS and len(offshore_clusters) > 0:
            offshore_clusters_group = fm.FeatureGroup(name="Offshore Clusters", show=False)
            fm.GeoJson(
                offshore_clusters,
                marker=fm.CircleMarker(
                    fill=True,
                    fill_opacity=0,
                ),
                tooltip=make_cluster_tooltip(),
                style_function=make_style_fn('yellow', 1),
            ).add_to(offshore_clusters_group)
            offshore_clusters_group.add_to(m)

    # ---------------------------------
    # other landsource pixels
    # ---------------------------------
    if "type" in wf.cleaned.columns:
        other_land_points = gdf_plot[gdf_plot["type"] == 2][
            ["geometry", "display_type", "frp", "datetime"]
            ]
        if len(other_land_points) < MAX_ROWS and len(other_land_points) > 0:
            other_land_pixels_group = fm.FeatureGroup(name="Other Land Source Pixels", show=False)
            fm.GeoJson(
                other_land_points,
                marker=fm.CircleMarker(radius=2,
                                    fill=True,
                                    fill_color="magenta",
                                    fill_opacity=1,
                                    weight=0
                                    ),
                tooltip=make_point_tooltip()
            ).add_to(other_land_pixels_group)
            other_land_pixels_group.add_to(m)
        other_land_clusters = cluster_gdf_plot[cluster_gdf_plot["type"] == 2][
            ["geometry", "display_type", "pixel_count", "frp_sum", "frp_mean", "cluster_size_approx", "time_span"]
            ]
        if len(other_land_clusters) < MAX_ROWS and len(other_land_clusters) > 0:
            other_land_clusters_group = fm.FeatureGroup(name="Other Land Source Clusters", show=False)
            fm.GeoJson(
                other_land_clusters,
                marker=fm.CircleMarker(
                    fill=True,
                    fill_opacity=0,
                ),
                tooltip=make_cluster_tooltip(),
                style_function=make_style_fn('magenta', 1),
            ).add_to(other_land_clusters_group)
            other_land_clusters_group.add_to(m)

    fm.LayerControl(collapsed=False,
                    position="topleft").add_to(m)
    plugins.MeasureControl(
        position="bottomleft",
        primary_length_unit="kilometers",
        secondary_length_unit="miles",
        primary_area_unit="sqmeters",
        secondary_area_unit="acres",
    ).add_to(m)

    if save == True:
        output_path = os.path.join(os.getcwd(), f"wildfire_{wf.area_display}_{wf.date}.html")
        m.save(output_path)
        webbrowser.open(f"file://{output_path}")

    return m

In [ ]:
map_wf(wf)